# Streaming RAG Responses

Non-streaming RAG makes the user stare at a blank screen until the full answer is ready — which can be several seconds for long responses. Streaming displays each token as it is generated, turning a frustrating wait into a fast-feeling experience. SynapseKit's `astream()` makes this a one-line change from `aquery()`.

**What you'll build:** A streaming RAG pipeline that prints tokens in real time, plus a FastAPI endpoint that streams answers via Server-Sent Events.

**Time:** ~10 min | **Difficulty:** Beginner

Guide: [Streaming RAG Responses](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/streaming-rag)

## Prerequisites & Setup

You will need:
- An **OpenAI API key** — set `OPENAI_API_KEY` in the env cell below
- `synapsekit` and optionally `fastapi` + `uvicorn` for the SSE endpoint

**What you'll learn:**
- How `astream()` differs from `aquery()` and when to use each
- How to print tokens to the terminal in real time
- How to wire streaming RAG into a FastAPI SSE endpoint
- Why retrieval still happens once before generation in streaming mode

In [ ]:
!pip install synapsekit fastapi uvicorn -q

In [ ]:
import os

# Set your OpenAI API key before running the cells below
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Build a Pipeline with Sample Documents

Set up the RAGPipeline and add documents. This is identical to the quickstart — streaming is a drop-in replacement for `aquery()`.

In [ ]:
import asyncio
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
    vectorstore=InMemoryVectorStore(),
)

docs = [
    "SynapseKit supports streaming via the astream() async generator method.",
    "Streaming lets the UI render partial output as soon as the first token arrives.",
    "The retrieval step happens once before generation; it is not streamed.",
    "FastAPI's StreamingResponse combined with SSE is the standard way to stream to browsers.",
]

await rag.aadd(docs)
print("Documents added.")

## Step 2: Stream to the Terminal

`astream()` returns an async generator that yields one token string at a time. `flush=True` ensures each token appears immediately rather than buffering in the terminal's stdout buffer.

`astream()` takes the same arguments as `aquery()` — including `k` for the number of retrieved chunks. The only difference is the return type: a string vs. an async generator.

In [ ]:
# astream() returns an async generator that yields one token string at a time.
# flush=True ensures each token appears immediately rather than buffering in
# the terminal's stdout buffer.
print("Answer: ", end="")
async for token in rag.astream("How does SynapseKit handle streaming?"):
    print(token, end="", flush=True)
print()  # newline when the stream finishes

## Step 3: Measure Time-to-First-Token

Time-to-first-token (TTFT) is the user-perceived latency before anything appears on screen. Streaming reduces TTFT from `total_generation_time` to `retrieval_time + first_token_time`, which is typically 3–5x faster.

In [ ]:
import time

# time-to-first-token (TTFT) is the user-perceived latency before anything
# appears on screen. Streaming reduces TTFT from total_generation_time to
# retrieval_time + first_token_time, which is typically 3-5x faster.
start = time.perf_counter()
first_token_time = None

async for token in rag.astream("What is time-to-first-token?"):
    if first_token_time is None:
        first_token_time = time.perf_counter() - start
        print(f"[TTFT: {first_token_time:.2f}s] ", end="")
    print(token, end="", flush=True)

print(f"\n[Total: {time.perf_counter() - start:.2f}s]")

## Step 4: Collect the Full Response While Streaming

Sometimes you need both the streamed output for UX and the complete string for logging or post-processing. Accumulate tokens into a list and join.

In [ ]:
# Sometimes you need both the streamed output for UX and the complete string
# for logging or post-processing. Accumulate tokens into a list and join.
tokens = []
async for token in rag.astream("What does astream() return?"):
    tokens.append(token)
    print(token, end="", flush=True)
print()

full_response = "".join(tokens)
print(f"\nFull response ({len(full_response)} chars): {full_response[:100]}...")

## Step 5: FastAPI SSE Endpoint

Server-Sent Events (SSE) is the standard browser protocol for streaming text from a server. Each event is a newline-terminated `"data: ..."` line. `StreamingResponse` with `media_type="text/event-stream"` handles the HTTP framing.

Run with `uvicorn app:app --reload` and open `http://localhost:8000/rag/stream?question=How+does+streaming+work`.

In [ ]:
# Server-Sent Events (SSE) is the standard browser protocol for streaming
# text from a server. Each event is a newline-terminated "data: ..." line.
# StreamingResponse with media_type="text/event-stream" handles the HTTP framing.

from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()

@app.get("/rag/stream")
async def stream_rag(question: str):
    async def event_generator():
        async for token in rag.astream(question):
            # SSE format: each message must start with "data: " and end with "\n\n"
            yield f"data: {token}\n\n"
        # Signal the client that the stream has ended.
        yield "data: [DONE]\n\n"

    return StreamingResponse(event_generator(), media_type="text/event-stream")

print("FastAPI app defined. Run with: uvicorn app:app --reload")
print("Then open: http://localhost:8000/rag/stream?question=How+does+streaming+work")

## Step 6: JavaScript Client for the SSE Endpoint

Paste this snippet into a browser console or an HTML file to test the SSE endpoint.

In [ ]:
# JavaScript snippet to consume the SSE stream in a browser
js_snippet = """
<script>
const source = new EventSource(
  "http://localhost:8000/rag/stream?question=How+does+streaming+work"
);

source.onmessage = (event) => {
  if (event.data === "[DONE]") {
    source.close();
    return;
  }
  // Append each token to the DOM as it arrives.
  document.getElementById("answer").textContent += event.data;
};
</script>
<div id="answer"></div>
"""
print(js_snippet)

## Complete Working Example

A single self-contained cell that builds the pipeline, streams a response, and reports token count.

In [ ]:
import asyncio
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
    vectorstore=InMemoryVectorStore(),
)

async def main():
    await rag.aadd([
        "SynapseKit supports streaming via the astream() async generator method.",
        "Streaming lets the UI render partial output as soon as the first token arrives.",
        "The retrieval step happens once before generation; it is not streamed.",
        "FastAPI's StreamingResponse combined with SSE is the standard way to stream to browsers.",
    ])

    print("Streaming response:")
    print("-" * 40)
    tokens = []
    async for token in rag.astream("How does SynapseKit handle streaming?"):
        tokens.append(token)
        print(token, end="", flush=True)
    print()
    print("-" * 40)
    print(f"Total tokens received: {len(tokens)}")

asyncio.run(main())

## Next Steps

- [RAG with Conversation Memory](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/rag-with-memory) — add multi-turn memory to a streaming pipeline
- [RAG in 3 Lines](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/quickstart-3-lines) — review the non-streaming baseline
- [RAGPipeline reference](https://synapsekit.github.io/synapsekit-docs/docs/rag/pipeline) — full `astream()` API documentation